In [0]:
spark.conf.set(
  "fs.azure.account.key.olistdatastoragepradeep.dfs.core.windows.net", 
  "storage-key"
)

In [0]:
base_path = "abfss://olistdata@olistdatastoragepradeep.dfs.core.windows.net"

bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"

In [0]:
from pyspark.sql.functions import col, to_timestamp, to_date, datediff

# Read from bronze (example)
# orders_df = spark.read.option("header","true").csv(f"{bronze_path}/olist_orders_dataset.csv")

orders_clean = (orders_df
    .dropDuplicates(["order_id"])
    # parse timestamps FIRST
    .withColumn("order_purchase_ts", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("order_delivered_customer_ts", to_timestamp(col("order_delivered_customer_date")))
    .withColumn("order_estimated_delivery_dt", to_date(col("order_estimated_delivery_date")))
    # derive date columns
    .withColumn("order_purchase_dt", to_date(col("order_purchase_ts")))
    .withColumn("order_delivered_customer_dt", to_date(col("order_delivered_customer_ts")))
    # metrics
    .withColumn("actual_delivery_days", datediff(col("order_delivered_customer_dt"), col("order_purchase_dt")))
    .withColumn("estimated_delivery_days", datediff(col("order_estimated_delivery_dt"), col("order_purchase_dt")))
    .withColumn("delay_days", col("actual_delivery_days") - col("estimated_delivery_days"))
)

# If you want to keep the raw string columns, don't drop them.
# If you want cleaner schema, drop them:
# orders_clean = orders_clean.drop("order_purchase_timestamp","order_delivered_customer_date","order_estimated_delivery_date")

(orders_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/orders"))

In [0]:
from pyspark.sql.functions import col, to_timestamp

items = spark.read.option("header","true").csv(f"{bronze_path}/olist_order_items_dataset.csv")

items_clean = (items
  .dropDuplicates(["order_id","order_item_id"])
  .withColumn("shipping_limit_ts", to_timestamp("shipping_limit_date"))
  .withColumn("price", col("price").cast("double"))
  .withColumn("freight_value", col("freight_value").cast("double"))
  .drop("shipping_limit_date")
)

(items_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .save(f"{silver_path}/order_items"))

In [0]:
display(items_clean)

silver/payments

In [0]:
from pyspark.sql.functions import col, trim

payments_df = spark.read.option("header","true").csv(f"{bronze_path}/olist_order_payments_dataset.csv")

payments_clean = (payments_df
    .dropDuplicates(["order_id", "payment_sequential"])   # correct grain
    .withColumn("payment_sequential", col("payment_sequential").cast("int"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
    .withColumn("payment_value", col("payment_value").cast("double"))
    .withColumn("payment_type", trim(col("payment_type")))
)

(payments_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/payments"))

Silver Customers (clean + dedupe)

In [0]:
from pyspark.sql.functions import col, trim

customers_df = spark.read.option("header","true").csv(f"{bronze_path}/olist_customers_dataset.csv")

customers_clean = (customers_df
    .dropDuplicates(["customer_id"])
    .withColumn("customer_unique_id", trim(col("customer_unique_id")))
    .withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast("int"))
    .withColumn("customer_city", trim(col("customer_city")))
    .withColumn("customer_state", trim(col("customer_state")))
)

(customers_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/customers"))

Silver Products (clean + typed + deduped)

In [0]:
from pyspark.sql.functions import col, trim

products_df = spark.read.option("header","true").csv(f"{bronze_path}/olist_products_dataset.csv")

products_clean = (products_df
    .dropDuplicates(["product_id"])
    .withColumn("product_category_name", trim(col("product_category_name")))
    .withColumn("product_name_lenght", col("product_name_lenght").cast("int"))
    .withColumn("product_description_lenght", col("product_description_lenght").cast("int"))
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int"))
    .withColumn("product_weight_g", col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", col("product_width_cm").cast("double"))
)

(products_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/products"))

silver/seller

In [0]:
from pyspark.sql.functions import col, trim

sellers_df = spark.read.option("header","true").csv(f"{bronze_path}/olist_sellers_dataset.csv")

sellers_clean = (sellers_df
    .dropDuplicates(["seller_id"])
    .withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("int"))
    .withColumn("seller_city", trim(col("seller_city")))
    .withColumn("seller_state", trim(col("seller_state")))
)

(sellers_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/sellers"))

silver/reviews

In [0]:
reviews_df = (spark.read
  .format("csv")
  .option("header", "true")
  .option("multiLine", "true")
  .option("quote", "\"")
  .option("escape", "\"")
  .option("mode", "PERMISSIVE")
  .load(f"{bronze_path}/olist_order_reviews_dataset.csv")
)


reviews_df.select("review_id","review_score","review_creation_date","review_answer_timestamp")
from pyspark.sql.functions import col, trim, to_timestamp, expr

reviews_clean = (reviews_df
  .dropDuplicates(["review_id"])
  .withColumn("review_score", expr("try_cast(review_score as int)"))
  .withColumn("review_creation_ts", to_timestamp(col("review_creation_date"), "yyyy-MM-dd HH:mm:ss"))
  .withColumn("review_answer_ts", to_timestamp(col("review_answer_timestamp"), "yyyy-MM-dd HH:mm:ss"))
  .withColumn("review_comment_title", trim(col("review_comment_title")))
  .withColumn("review_comment_message", trim(col("review_comment_message")))
  .drop("review_creation_date", "review_answer_timestamp")
)

(reviews_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/reviews"))


silver/geolocation

In [0]:
from pyspark.sql.functions import col, trim, avg

geo_df = spark.read.option("header","true").csv(f"{bronze_path}/olist_geolocation_dataset.csv")

geo_clean = (geo_df
    .withColumn("geolocation_zip_code_prefix", col("geolocation_zip_code_prefix").cast("int"))
    .withColumn("geolocation_lat", col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", col("geolocation_lng").cast("double"))
    .withColumn("geolocation_city", trim(col("geolocation_city")))
    .withColumn("geolocation_state", trim(col("geolocation_state")))
)

geo_zip = (geo_clean
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("zip_lat"),
        avg("geolocation_lng").alias("zip_lng")
    )
)

(geo_zip.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/geolocation_zip"))

In [0]:
!pip install pymongo

from pymongo import MongoClient
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit, trim, col
uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database
db = client[database]
collection = db["product_categories"]

# Pull docs (small table)
pdf = pd.DataFrame(list(collection.find({}, {"_id": 0})))

# Pandas -> Spark
sdf = spark.createDataFrame(pdf)

# Basic Silver cleaning + audit columns
silver_df = (sdf
    .dropDuplicates()
    .select([trim(col(c)).alias(c) for c in sdf.columns])   # trim all strings
    .withColumn("_ingestion_time", current_timestamp())
    .withColumn("_source", lit("mongo"))
)

# Write straight to Silver
(silver_df.write.format("delta")
 .mode("overwrite")                 # use overwrite for small dimension
 .option("overwriteSchema", "true")
 .save(f"{silver_path}/product_categories"))

In [0]:
mongo_data